# 2333. 最小差值平方和

先取绝对差值，预算合并为 k1+k2。把差值看作柱高，每次优先降低最高柱；排序后按整组降层，避免逐次模拟。

[原文](https://leetcode.cn/problems/minimum-sum-of-squared-difference/solution/2333zhe-ye-neng-jiao-mediumcong-liang-ba-50lm/) · 整理于 2026-10-17。原代码保持原样；补充注释另放单元格。

## 排序后整组削峰

最高组大小为 i，从 height 降到下一层的成本是 `i*(height-next_height)`。预算不足时 `q,r=divmod(k,i)`，其中 r 根再多减 1。

In [ ]:
# 单次把 d 降为 d-1，平方和减少 2*d-1，所以先处理最大差值。
# ordered_diff 原数组不必反复改写；i 表示已合并到当前高度的前缀大小。
# 预算不足时贡献为 (i-r)*(height-q)^2 + r*(height-q-1)^2。
# 追加的 0 是高度哨兵；k >= sum(diff) 时可直接归零，不要求用完预算。

In [ ]:
from typing import List


class Solution:
    def minSumSquareDiff(
        self,
        nums1: List[int],
        nums2: List[int],
        k1: int,
        k2: int
    ) -> int:
        diff = []
        for i in range(len(nums1)):
            diff.append(abs(nums1[i] - nums2[i]))

        k = k1 + k2

        # 操作次数足够，让所有差值归零
        if k >= sum(diff):
            return 0

        ordered_diff = sorted(diff, reverse=True)
        ordered_diff.append(0)  # 最后一组也可以降低到 0

        for i in range(1, len(ordered_diff)):
            # 此时前 i 个差值都已降低到 ordered_diff[i - 1]
            height = ordered_diff[i - 1]
            tmp_diff = height - ordered_diff[i]

            # 前 i 个差值一起降低 tmp_diff，需要的操作次数
            cost = i * tmp_diff

            if k >= cost:
                k -= cost
            else:
                # 预算不足以降低到下一层
                q, r = divmod(k, i)
                final_height = height - q

                # i-r 个降低 q，r 个额外再降低 1
                ans = (
                    (i - r) * final_height ** 2
                    + r * (final_height - 1) ** 2
                )

                # 后面的差值没有被修改
                for j in range(i, len(ordered_diff)):
                    ans += ordered_diff[j] ** 2

                return ans

        return 0

In [ ]:
assert Solution().minSumSquareDiff([8, 8, 5, 2], [0, 0, 0, 0], 11, 0) == 38
assert Solution().minSumSquareDiff([1], [3], 1, 0) == 1
assert Solution().minSumSquareDiff([1, 2], [1, 2], 0, 0) == 0
assert Solution().minSumSquareDiff([4], [0], 9, 0) == 0

## 旧版瓶颈

设有效操作次数为 T。反复排序的逐次模拟上界 O(T·n log n)；只排序一次再插回列表仍可能 O(n log n + T·n)。列表 `pop(0)`、插入与切片都可能移动或复制元素。原文仅公开下面的局部片段，未提供两版完整实现。

```python
while k > 0:
    ordered_diff = sorted(ordered_diff, reverse=True)
    top_var = ordered_diff.pop(0)
    # 将最大差值减 1，再放回去
```

单元素列表弹出后再访问 `[0]` 会越界。原文边界片段：

```python
top_var = ordered_diff.pop(0)
# ...
ordered_diff[0]
```

最终版排序 O(n log n)，降层扫描与一次尾部求和 O(n)，额外空间 O(n)。`k == cost` 应完整降层；重复高度自然通过 cost=0 合并。